In [21]:
print("Feature Enginnering Notebook")

Feature Enginnering Notebook


# Import Library

In [43]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load Data

In [23]:
DATA_PATH = "../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv"
df = pd.read_csv(DATA_PATH)
df

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0
...,...,...,...,...,...,...
1165,NaN,NaN,NaN,NaN,NaN,NaN
1166,NaN,NaN,NaN,NaN,NaN,NaN
1167,NaN,NaN,NaN,NaN,NaN,NaN
1168,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

In [25]:
df

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,33.0,53.0,34.0,"6,566",436.0
1,2023-01-22,32.0,49.0,39.0,"7,122",227.0
2,2023-01-23,32.0,50.0,39.0,"7,280",181.0
3,2023-01-24,47.0,42.0,47.0,"7,433",175.0
4,2023-01-25,20.0,22.0,41.0,"7,538",180.0
...,...,...,...,...,...,...
1165,NaT,NaN,NaN,NaN,NaN,NaN
1166,NaT,NaN,NaN,NaN,NaN,NaN
1167,NaT,NaN,NaN,NaN,NaN,NaN
1168,NaT,NaN,NaN,NaN,NaN,NaN


# Rename Columns for easy readability

In [26]:
df.columns = [
    "date",
    "cbp_daily_intake",
    "cbp_custody",
    "cbp_transfers_to_hhs",
    "hhs_care",
    "hhs_discharges",
]

In [27]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges
0,2023-01-12,33.0,53.0,34.0,"6,566",436.0
1,2023-01-22,32.0,49.0,39.0,"7,122",227.0
2,2023-01-23,32.0,50.0,39.0,"7,280",181.0
3,2023-01-24,47.0,42.0,47.0,"7,433",175.0
4,2023-01-25,20.0,22.0,41.0,"7,538",180.0
...,...,...,...,...,...,...
1165,NaT,NaN,NaN,NaN,NaN,NaN
1166,NaT,NaN,NaN,NaN,NaN,NaN
1167,NaT,NaN,NaN,NaN,NaN,NaN
1168,NaT,NaN,NaN,NaN,NaN,NaN


# Changing Datatypes

In [28]:
df.dtypes

date                    datetime64[ns]
cbp_daily_intake               float64
cbp_custody                    float64
cbp_transfers_to_hhs           float64
hhs_care                        object
hhs_discharges                 float64
dtype: object

In [29]:
numeric_cols = [
    "cbp_daily_intake",
    "cbp_custody",
    "cbp_transfers_to_hhs",
    "hhs_care",
    "hhs_discharges",
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("—", "", regex=False)
        .str.replace("NA", "", regex=False)
        .str.replace("N/A", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [30]:
df.dtypes

date                    datetime64[ns]
cbp_daily_intake               float64
cbp_custody                    float64
cbp_transfers_to_hhs           float64
hhs_care                       float64
hhs_discharges                 float64
dtype: object

# Null value checking

In [31]:
df.isnull().sum()

date                    450
cbp_daily_intake        450
cbp_custody             450
cbp_transfers_to_hhs    450
hhs_care                450
hhs_discharges          450
dtype: int64

In [32]:
# Drop rows where date itself is missing
df = df.dropna(subset=["date"])

In [33]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0
...,...,...,...,...,...,...
715,2025-12-15,11.0,42.0,9.0,2470.0,7.0
716,2025-12-16,8.0,54.0,15.0,2468.0,9.0
717,2025-12-17,7.0,31.0,11.0,2481.0,10.0
718,2025-12-18,11.0,50.0,6.0,2472.0,16.0


In [34]:
df.isnull().sum()

date                    0
cbp_daily_intake        0
cbp_custody             0
cbp_transfers_to_hhs    0
hhs_care                0
hhs_discharges          0
dtype: int64

# Total System Load (MOST IMPORTANT)

* Meaning:
“System pe total responsibility kitni hai?”

In [35]:
df["total_system_load"] = df["cbp_custody"] + df["hhs_care"]

In [36]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges,total_system_load
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0,7171.0
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0,7330.0
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0,7475.0
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0,7560.0
...,...,...,...,...,...,...,...
715,2025-12-15,11.0,42.0,9.0,2470.0,7.0,2512.0
716,2025-12-16,8.0,54.0,15.0,2468.0,9.0,2522.0
717,2025-12-17,7.0,31.0,11.0,2481.0,10.0,2512.0
718,2025-12-18,11.0,50.0,6.0,2472.0,16.0,2522.0


# Net Daily Intake (Pressure Indicator)

* Interpretation: > 0 → pressure build & < 0 → relief phase

Ye policy-level metric hai

In [37]:
df["net_daily_intake"] = df["cbp_transfers_to_hhs"] - df["hhs_discharges"]

In [38]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges,total_system_load,net_daily_intake
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,-402.0
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0,7171.0,-188.0
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0,7330.0,-142.0
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0,7475.0,-128.0
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0,7560.0,-139.0
...,...,...,...,...,...,...,...,...
715,2025-12-15,11.0,42.0,9.0,2470.0,7.0,2512.0,2.0
716,2025-12-16,8.0,54.0,15.0,2468.0,9.0,2522.0,6.0
717,2025-12-17,7.0,31.0,11.0,2481.0,10.0,2512.0,1.0
718,2025-12-18,11.0,50.0,6.0,2472.0,16.0,2522.0,-10.0


# Care Load Growth Rate (Day-over-Day)

In [39]:
df["care_load_growth_rate"] = df["total_system_load"].pct_change() * 100

In [40]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges,total_system_load,net_daily_intake,care_load_growth_rate
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,-402.0,NaN
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0,7171.0,-188.0,8.339628
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0,7330.0,-142.0,2.217264
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0,7475.0,-128.0,1.978172
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0,7560.0,-139.0,1.137124
...,...,...,...,...,...,...,...,...,...
715,2025-12-15,11.0,42.0,9.0,2470.0,7.0,2512.0,2.0,0.600721
716,2025-12-16,8.0,54.0,15.0,2468.0,9.0,2522.0,6.0,0.398089
717,2025-12-17,7.0,31.0,11.0,2481.0,10.0,2512.0,1.0,-0.396511
718,2025-12-18,11.0,50.0,6.0,2472.0,16.0,2522.0,-10.0,0.398089


# Backlog Indicator (Sustained Pressure)

In [41]:
df["backlog_indicator"] = df["net_daily_intake"] > 0

In [42]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges,total_system_load,net_daily_intake,care_load_growth_rate,backlog_indicator
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,-402.0,NaN,False
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0,7171.0,-188.0,8.339628,False
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0,7330.0,-142.0,2.217264,False
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0,7475.0,-128.0,1.978172,False
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0,7560.0,-139.0,1.137124,False
...,...,...,...,...,...,...,...,...,...,...
715,2025-12-15,11.0,42.0,9.0,2470.0,7.0,2512.0,2.0,0.600721,True
716,2025-12-16,8.0,54.0,15.0,2468.0,9.0,2522.0,6.0,0.398089,True
717,2025-12-17,7.0,31.0,11.0,2481.0,10.0,2512.0,1.0,-0.396511,True
718,2025-12-18,11.0,50.0,6.0,2472.0,16.0,2522.0,-10.0,0.398089,False


# Discharge Offset Ratio

In [44]:
df["discharge_offset_ratio"] = np.where(
    df["cbp_transfers_to_hhs"] > 0,
    df["hhs_discharges"] / df["cbp_transfers_to_hhs"],
    np.nan,
)

In [45]:
df

,date,cbp_daily_intake,cbp_custody,cbp_transfers_to_hhs,hhs_care,hhs_discharges,total_system_load,net_daily_intake,care_load_growth_rate,backlog_indicator,discharge_offset_ratio
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,-402.0,NaN,False,12.823529
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0,7171.0,-188.0,8.339628,False,5.820513
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0,7330.0,-142.0,2.217264,False,4.641026
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0,7475.0,-128.0,1.978172,False,3.723404
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0,7560.0,-139.0,1.137124,False,4.390244
...,...,...,...,...,...,...,...,...,...,...,...
715,2025-12-15,11.0,42.0,9.0,2470.0,7.0,2512.0,2.0,0.600721,True,0.777778
716,2025-12-16,8.0,54.0,15.0,2468.0,9.0,2522.0,6.0,0.398089,True,0.600000
717,2025-12-17,7.0,31.0,11.0,2481.0,10.0,2512.0,1.0,-0.396511,True,0.909091
718,2025-12-18,11.0,50.0,6.0,2472.0,16.0,2522.0,-10.0,0.398089,False,2.666667


# KPI Snapshot (Executive Friendly)

In [46]:
kpi_summary = {
    "avg_total_system_load": df["total_system_load"].mean(),
    "max_system_load": df["total_system_load"].max(),
    "avg_net_daily_intake": df["net_daily_intake"].mean(),
    "max_daily_growth_rate_%": df["care_load_growth_rate"].max(),
    "avg_discharge_offset_ratio": df["discharge_offset_ratio"].mean(),
}

pd.DataFrame.from_dict(kpi_summary, orient="index", columns=["value"])

,value
avg_total_system_load,6232.769444
max_system_load,11762.000000
avg_net_daily_intake,-44.738889
max_daily_growth_rate_%,8.339628
avg_discharge_offset_ratio,2.013512


# Normal Sanity Check

In [47]:
df[["date", "cbp_custody", "hhs_care", "total_system_load", "net_daily_intake"]].head()

,date,cbp_custody,hhs_care,total_system_load,net_daily_intake
0,2023-01-12,53.0,6566.0,6619.0,-402.0
1,2023-01-22,49.0,7122.0,7171.0,-188.0
2,2023-01-23,50.0,7280.0,7330.0,-142.0
3,2023-01-24,42.0,7433.0,7475.0,-128.0
4,2023-01-25,22.0,7538.0,7560.0,-139.0


# This is my clean data saved into csv file for Streamlit 

In [49]:
df.to_csv("../data/processed/clean_children_data.csv", index=False)
print("Clean Data Saved Successfully.....")

Clean Data Saved Successfully.....
